In [1]:
import pandas as pd
import os
import re
from spellchecker import SpellChecker
import concurrent.futures


In [2]:
lookup_df = pd.read_csv('../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv')

spell = SpellChecker(language=None)
DICT_PATH = '../Code Tambahan/indonesian_dict.json' 
# DICT_PATH = '../Code Tambahan/indonesian_dict_strategy_B.json' 
# DICT_PATH = '../Code Tambahan/indonesian_dict_strategy_C.json' 

if os.path.exists(DICT_PATH):
    spell.word_frequency.load_dictionary(DICT_PATH)
    print(f"Loaded JSON dictionary successfully.")
else:
    raise FileNotFoundError(
        f"Custom dictionary not found at '{DICT_PATH}'. "
        "Check that the working directory is correct before running."
    )

Loaded JSON dictionary successfully.


In [3]:
def get_is_from_indonesia(input_path):
    filename = os.path.basename(input_path)
    try:
        file_id = filename.split('_')[0].strip()
    except IndexError:
        return False, -1

    for index, row in lookup_df.iterrows():
        raw_kamus = str(row['kamus'])
        match = re.search(r'\d+', raw_kamus)
        dict_id = match.group() if match else raw_kamus.strip()
            
        if dict_id == file_id:
            return True, int(row['is_from_indonesia'])
            
    return False, -1

In [4]:
def fix_sentence(sentence):
    def replace(match):
        word = match.group(0)
        lower_word = word.lower()
        
        if lower_word in spell.unknown([lower_word]):
            correction = spell.correction(lower_word)
            if correction and correction != lower_word:
                if word.istitle():
                    return correction.capitalize()
                elif word.isupper():
                    return correction.upper()
                else:
                    return correction
        return word

    return re.sub(r'[A-Za-z]+', replace, str(sentence))

In [ ]:
def process_file(file_name):
    try:
        input_file  = os.path.join('../Ekstraksi/11. Parallel Corpus - Fixed', file_name)
        output_file = os.path.join('../Ekstraksi/12. Parallel Corpus - Spelling Checker', file_name)

        success, is_from_indonesia = get_is_from_indonesia(input_file)
        if not success:
            return f"Skip: No matching ID for {file_name}"

        df = pd.read_csv(input_file)
        kalimat_asal_result_list   = []
        kalimat_tujuan_result_list = []

        for index, row in df.iterrows():
            orig_asal   = str(row['kalimat_asal'])
            orig_tujuan = str(row['kalimat_tujuan'])

            if is_from_indonesia:
                kalimat_asal_result_list.append(fix_sentence(orig_asal))
                kalimat_tujuan_result_list.append(orig_tujuan)
            else:
                kalimat_asal_result_list.append(orig_asal)
                kalimat_tujuan_result_list.append(fix_sentence(orig_tujuan))

        result_df = pd.DataFrame({
            'kalimat_asal':   kalimat_asal_result_list,
            'kalimat_tujuan': kalimat_tujuan_result_list
        })
        result_df.to_csv(output_file, index=False)
        return f"Processed: {file_name} ({len(result_df)} items)"

    except Exception as e:
        return f"Error in {file_name}: {str(e)}"

dir_path   = '../Ekstraksi/11. Parallel Corpus - Fixed'
output_dir = '../Ekstraksi/12. Parallel Corpus - Spelling Checker/'
os.makedirs(output_dir, exist_ok=True)

files = [f for f in os.listdir(dir_path)
         if f.endswith('.csv') and f != 'LookupIsFromIndonesia.csv']

print(f"Starting to process {len(files)} files...")
futures = []
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    for f in files:
        futures.append(executor.submit(process_file, f))
    for count, future in enumerate(concurrent.futures.as_completed(futures), 1):
        print(f"[{count}/{len(files)}] {future.result()}")

Starting to process 138 files...
[1/138] Processed: 10_Parcor.csv (2942 items)
[2/138] Processed: 10_Parcor_audit.csv (2942 items)
[3/138] Processed: 11_Parcor_audit.csv (609 items)
[4/138] Processed: 11_Parcor.csv (609 items)
[5/138] Processed: 14_Parcor.csv (997 items)
[6/138] Processed: 14_Parcor_audit.csv (997 items)
[7/138] Processed: 13_Parcor_audit.csv (588 items)
[8/138] Processed: 13_Parcor.csv (588 items)
[9/138] Processed: 12_Parcor_audit.csv (4810 items)
[10/138] Processed: 16_Parcor.csv (1889 items)
[11/138] Processed: 18_Parcor.csv (2341 items)
[12/138] Processed: 12_Parcor.csv (4810 items)
[13/138] Processed: 18_Parcor_audit.csv (2341 items)
[14/138] Processed: 17_Parcor_audit.csv (1673 items)
[15/138] Processed: 17_Parcor.csv (1673 items)
[16/138] Processed: 20_Parcor.csv (1719 items)
[17/138] Processed: 20_Parcor_audit.csv (1719 items)
[18/138] Processed: 16_Parcor_audit.csv (1889 items)
[19/138] Processed: 19_Parcor.csv (5579 items)
[20/138] Processed: 19_Parcor_audit